# 12 - Final Comparison

## Objective

Menyatukan evidence dari tahap 07 sampai 11 untuk memilih pipeline entity resolution yang paling rasional, dengan perhatian khusus pada runtime dan scalability.

## Important evaluation boundary

- Kualitas supervised hanya diukur pada 107 reviewed pairs.
- Candidate artifact blocking berisi 70.381 unique pairs.
- Baseline naive N x N untuk 50.000 row adalah 1.249.975.000 pairs dan tidak dijalankan.
- Runtime benchmark pada notebook ini adalah runtime lokal untuk snapshot data, bukan jaminan production SLA.

## Decision question

Apakah pipeline perlu menjalankan perbandingan seluruh dataset? Jawaban yang diuji di sini: tidak, karena blocking mengurangi pasangan menjadi candidate set yang finite dan dapat diukur.

In [ ]:
from pathlib import Path
from time import perf_counter

import pandas as pd

DATA_CANDIDATES = [
    Path.cwd() / 'data' / 'processed' / 'crm_50000_customers_standardized.csv',
    Path.cwd().parent / 'data' / 'processed' / 'crm_50000_customers_standardized.csv',
]
DATA_PATH = next((path.resolve() for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Dataset terstandardisasi tidak ditemukan.')

PROCESSED_DIR = DATA_PATH.parent
REQUIRED_FILES = {
    'rule_quality': PROCESSED_DIR / 'benchmark_rule_quality.csv',
    'blocking_resources': PROCESSED_DIR / 'benchmark_blocking_resources.csv',
    'pair_artifact': PROCESSED_DIR / 'benchmark_pair_artifact.csv',
    'review_queue': PROCESSED_DIR / 'manual_review_queue.csv',
}
missing_files = [str(path) for path in REQUIRED_FILES.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(f'Artifact final comparison belum tersedia: {missing_files}')

rule_quality = pd.read_csv(REQUIRED_FILES['rule_quality'])
blocking_resources = pd.read_csv(REQUIRED_FILES['blocking_resources'])
pair_artifact = pd.read_csv(REQUIRED_FILES['pair_artifact'])
review_queue = pd.read_csv(REQUIRED_FILES['review_queue'], sep=';')

df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
print('Reviewed pairs:', len(review_queue))
print('Blocking artifact rows:', len(pair_artifact))

## Experiment 1 - Consolidate quality and scalability evidence

Kualitas rule dan scalability disimpan sebagai tabel terpisah agar tidak mencampur metrik sample dengan metrik computational.

In [ ]:
n_rows = len(df)
naive_pair_count = n_rows * (n_rows - 1) // 2
pair_artifact_values = pair_artifact.set_index('metric')['value']
unique_candidate_pairs = int(pair_artifact_values['unique_pair_rows'])

quality_comparison = rule_quality[[
    'rule', 'labeled_rows', 'true_positive', 'false_positive',
    'false_negative', 'true_negative', 'precision', 'recall', 'f1',
]].copy()
quality_comparison['scope'] = '107 reviewed pairs only'

scalability_summary = pd.DataFrame({
    'metric': [
        'dataset_rows',
        'naive_pair_count',
        'blocking_artifact_rows',
        'unique_candidate_pairs',
        'candidate_reduction_vs_naive_percentage',
        'reviewed_pairs',
    ],
    'value': [
        n_rows,
        naive_pair_count,
        int(pair_artifact_values['artifact_rows']),
        unique_candidate_pairs,
        (1 - unique_candidate_pairs / naive_pair_count) * 100,
        len(review_queue),
    ],
})

quality_comparison, scalability_summary

## Experiment 2 - Runtime feasibility without N x N execution

Candidate artifact dibaca dan di-deduplicate sebagai proxy biaya downstream. Full pairwise comparison tidak dijalankan karena baseline-nya 1,25 miliar pair.

In [ ]:
start_time = perf_counter()
loaded_pairs = pd.read_csv(
    PROCESSED_DIR / 'blocking_candidate_pairs.csv',
    usecols=['left_row_index', 'right_row_index', 'blocking_strategy'],
)
unique_pairs = loaded_pairs[['left_row_index', 'right_row_index']].drop_duplicates()
artifact_runtime_seconds = perf_counter() - start_time

runtime_summary = pd.DataFrame({
    'operation': [
        'load_and_deduplicate_blocking_artifact',
        'naive_full_pairwise_comparison',
    ],
    'pair_count': [len(loaded_pairs), naive_pair_count],
    'runtime_seconds': [artifact_runtime_seconds, pd.NA],
    'status': [
        'measured_on_current_environment',
        'not_run_to_avoid_excessive_computation',
    ],
})
runtime_summary

## Experiment 3 - Final method comparison

Keputusan mempertimbangkan precision risk, coverage diagnostic, candidate reduction, interpretability, dan runtime. `reference coverage` bukan recall karena tidak berasal dari ground truth.

In [ ]:
final_method_comparison = pd.DataFrame([
    {
        'method': 'agreement_count >= 4',
        'strength': 'highest precision in reviewed sample; transparent',
        'risk': 'recall outside reviewed sample is unknown',
        'scalability': 'high after blocking',
        'decision': 'candidate high-confidence for review, not automatic merge',
    },
    {
        'method': 'agreement_count >= 2',
        'strength': 'higher candidate coverage than strict rule',
        'risk': '7 false positives in reviewed sample; DOB/city collisions',
        'scalability': 'high after blocking',
        'decision': 'do not use as automatic merge rule',
    },
    {
        'method': 'phone_prefix_7 blocking',
        'strength': '5,756 candidates and 77.53% reference coverage in prior snapshot',
        'risk': 'reference coverage is not recall; prefix collisions remain',
        'scalability': 'high for current candidate volume',
        'decision': 'retain as candidate-generation strategy to validate further',
    },
    {
        'method': 'email_domain blocking',
        'strength': 'simple key',
        'risk': 'candidate explosion to 399,502,708 pairs',
        'scalability': 'poor as a standalone key',
        'decision': 'reject as standalone blocking strategy',
    },
])
final_method_comparison

# Final Decision

## Apakah run terlalu lama?

- Full naive pairwise comparison: tidak perlu dijalankan. Jumlah pasangan teoritis terlalu besar.
- Blocking pipeline: layak dijalankan karena hanya memproses candidate pairs, bukan seluruh N x N.
- Fuzzy/probabilistic scoring: jalankan hanya pada candidate pairs hasil blocking dan ukur runtime aktual setiap snapshot.

## Recommended pipeline

1. Standardize raw data ke kolom turunan.
2. Generate candidates dengan beberapa blocking key high-recall.
3. Deduplicate candidate pair.
4. Hitung comparison vector/fuzzy score hanya pada candidate pair.
5. Kirim high-confidence candidate ke review atau proses lanjutan.
6. Jangan automatic merge sebelum validation set lebih luas tersedia.

## Limitations

- Metrik kualitas hanya berasal dari 107 reviewed pairs.
- Runtime artifact bukan runtime seluruh fuzzy/probabilistic pipeline.
- Tidak ada ground truth untuk mengukur candidate recall sebenarnya.
- Nilai reference coverage tidak boleh dilaporkan sebagai recall.

## Next step

Pipeline laboratory sudah memiliki keputusan awal. Pekerjaan berikutnya adalah memperluas manual validation, memperbaiki candidate strategy bila error analysis menemukan pola baru, lalu mempertimbangkan model probabilistic setelah label cukup.

In [ ]:
OUTPUT_DIR = DATA_PATH.parent
QUALITY_OUTPUT_PATH = OUTPUT_DIR / 'final_quality_comparison.csv'
SCALABILITY_OUTPUT_PATH = OUTPUT_DIR / 'final_scalability_summary.csv'
RUNTIME_OUTPUT_PATH = OUTPUT_DIR / 'final_runtime_summary.csv'
METHOD_OUTPUT_PATH = OUTPUT_DIR / 'final_method_comparison.csv'

quality_comparison.to_csv(QUALITY_OUTPUT_PATH, index=False)
scalability_summary.to_csv(SCALABILITY_OUTPUT_PATH, index=False)
runtime_summary.to_csv(RUNTIME_OUTPUT_PATH, index=False)
final_method_comparison.to_csv(METHOD_OUTPUT_PATH, index=False)

print('Saved:', QUALITY_OUTPUT_PATH)
print('Saved:', SCALABILITY_OUTPUT_PATH)
print('Saved:', RUNTIME_OUTPUT_PATH)
print('Saved:', METHOD_OUTPUT_PATH)
print('Raw dataset still exists:', (DATA_PATH.parents[1] / 'raw' / 'crm_50000_customers_dirty_v3.csv').exists())